In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe


# =============================================================================
# Task 3 — Population totals by ADM2 (WorldPop)
# Outputs (all prefixed with task3_):
#   tables/  : CSV
#   outputs/ : JSON
#   figures/ : PNGs
# =============================================================================

# ----------------------------
# Project structure
# ----------------------------
ROOT = Path(r"C:\Users\am636\copperbelt_worldpop_smod")

DATA = ROOT / "data_raw"
OUT = ROOT / "outputs"
TABDIR = ROOT / "tables"
FIGDIR = ROOT / "figures"

for d in (DATA, OUT, TABDIR, FIGDIR):
    d.mkdir(parents=True, exist_ok=True)

PREFIX = "task3_"

boundary_path = DATA / "ZMB_adm2_gadm41_Copperbelt.shp"
pop_path = DATA / "zmb_ppp_2020_constrained.tif"

missing = [p for p in (boundary_path, pop_path) if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required input file(s) in data_raw:\n" + "\n".join(str(p) for p in missing))

# Controls
ALL_TOUCHED = False

run_time_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")


# ----------------------------
# Output paths
# ----------------------------
csv_out = TABDIR / f"{PREFIX}pop_by_adm2.csv"
json_out = OUT / f"{PREFIX}pop_by_adm2.json"
map_png = FIGDIR / f"{PREFIX}pop_choropleth.png"
bar_png = FIGDIR / f"{PREFIX}pop_ranked_bar.png"


# ----------------------------
# Rasterised zonal sum (no extra dependencies)
# ----------------------------
def zonal_sum_rasterised(gdf: gpd.GeoDataFrame, raster_path: Path, nodata, all_touched: bool):
    """
    Computes per-polygon sum of raster values using rasterio.features.rasterize.
    Returns: np.ndarray of sums aligned to gdf row order.
    """
    from rasterio.features import rasterize

    sums = np.zeros(len(gdf), dtype="float64")

    with rasterio.open(raster_path) as src:
        arr = src.read(1, masked=False)
        transform = src.transform

        if nodata is not None:
            data = np.where(arr == nodata, 0.0, arr).astype("float64")
        else:
            data = arr.astype("float64")
            data = np.where(np.isfinite(data), data, 0.0)

        for i, geom in enumerate(gdf.geometry):
            if geom is None or geom.is_empty:
                sums[i] = 0.0
                continue

            mask = rasterize(
                [(geom, 1)],
                out_shape=data.shape,
                transform=transform,
                fill=0,
                all_touched=all_touched,
                dtype="uint8",
            )
            sums[i] = float((data * mask).sum())

    return sums


# ----------------------------
# 1) Read inputs and align CRS
# ----------------------------
gdf = gpd.read_file(boundary_path)

if gdf.crs is None:
    raise ValueError("Boundary CRS is missing.")

with rasterio.open(pop_path) as src:
    pop_crs = src.crs
    pop_nodata = src.nodata
    pop_dtype = src.dtypes[0]

if str(gdf.crs) != str(pop_crs):
    gdf = gdf.to_crs(pop_crs)

name_field = "NAME_2" if "NAME_2" in gdf.columns else None
if name_field is None:
    raise ValueError("NAME_2 not found in boundary attributes.")

print("Inputs:")
print(" - boundary:", boundary_path.name, "| features:", len(gdf), "| CRS:", gdf.crs)
print(" - population raster:", pop_path.name, "| CRS:", pop_crs, "| nodata:", pop_nodata, "| dtype:", pop_dtype)
print(" - all_touched:", ALL_TOUCHED)
print()


# ----------------------------
# 2) Zonal sum per ADM2
# ----------------------------
pop_sums = zonal_sum_rasterised(gdf, pop_path, pop_nodata, ALL_TOUCHED)

gdf["pop_sum"] = pop_sums.astype("float64")

out_tbl = gdf[[name_field]].copy()
out_tbl["pop_sum"] = gdf["pop_sum"].values
out_tbl["pop_sum_round"] = out_tbl["pop_sum"].round(0).astype("int64")
out_tbl = out_tbl.sort_values("pop_sum", ascending=False).reset_index(drop=True)
out_tbl["rank_desc"] = np.arange(1, len(out_tbl) + 1)

highest = out_tbl.iloc[0]
lowest = out_tbl.iloc[-1]
province_total = float(out_tbl["pop_sum"].sum())

print("Results:")
print(f" - Highest population ADM2: {highest[name_field]} | pop_sum ≈ {highest['pop_sum_round']:,}")
print(f" - Lowest population ADM2 : {lowest[name_field]} | pop_sum ≈ {lowest['pop_sum_round']:,}")
print(f" - Sum over ADM2 polygons: ≈ {province_total:,.0f}")
print()


# ----------------------------
# 3) Save outputs (CSV + JSON)
# ----------------------------
out_tbl.to_csv(csv_out, index=False)

summary = {
    "run_time_utc": run_time_utc,
    "inputs": {
        "boundary_path": str(boundary_path),
        "population_raster": str(pop_path),
        "boundary_crs": str(gdf.crs),
        "population_crs": str(pop_crs),
        "population_nodata": pop_nodata,
        "all_touched": ALL_TOUCHED,
        "method": "rasterio.features.rasterize (per-feature mask, sum of masked raster values)"
    },
    "results": {
        "highest": {
            "name": str(highest[name_field]),
            "pop_sum": float(highest["pop_sum"]),
            "pop_sum_round": int(highest["pop_sum_round"]),
        },
        "lowest": {
            "name": str(lowest[name_field]),
            "pop_sum": float(lowest["pop_sum"]),
            "pop_sum_round": int(lowest["pop_sum_round"]),
        },
        "province_total_pop_sum": province_total,
    },
}

with open(json_out, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved:")
print(" -", csv_out)
print(" -", json_out)
print()


# ----------------------------
# 4) Choropleth (raw totals) + labels for all districts
# ----------------------------
gdf_plot = gdf.copy()
gdf_plot["pop_sum_plot"] = gdf_plot["pop_sum"]

fig, ax = plt.subplots(figsize=(8, 7))

gdf_plot.plot(
    column="pop_sum_plot",
    ax=ax,
    legend=True,
    linewidth=0.8,
    edgecolor="black",
)

ax.set_title("ADM2 total population (WorldPop 2020 constrained)")
ax.set_axis_off()

for _, row in gdf_plot.iterrows():
    nm = str(row[name_field])
    pt = row.geometry.representative_point()
    ax.text(
        pt.x, pt.y, nm,
        fontsize=9,
        ha="center", va="center",
        color="black",
        path_effects=[pe.withStroke(linewidth=3, foreground="white")],
        bbox=dict(facecolor="white", alpha=0.55, edgecolor="none", pad=1.5),
    )

plt.tight_layout()
plt.savefig(map_png, dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved map:", map_png)


# ----------------------------
# 5) Ranked bar chart (annotation top-right)
# ----------------------------
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(out_tbl[name_field], out_tbl["pop_sum"])
ax.set_ylabel("Total population (sum of raster values)")
ax.set_title("Total population by ADM2 (WorldPop 2020 constrained)")
ax.tick_params(axis="x", labelrotation=60)

ax.text(
    0.99, 0.98,
    f"Highest: {highest[name_field]} ({highest['pop_sum_round']:,})\n"
    f"Lowest: {lowest[name_field]} ({lowest['pop_sum_round']:,})",
    transform=ax.transAxes,
    ha="right", va="top",
    fontsize=9,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=3),
)

plt.tight_layout()
plt.savefig(bar_png, dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved bar chart:", bar_png)


# ----------------------------
# 6) Short run note (console)
# ----------------------------
print("\nTask 3 note:")
print(f"Computed zonal sums of WorldPop population within ADM2 polygons (NoData={pop_nodata}, all_touched={ALL_TOUCHED}).")
print(f"Highest ADM2: {highest[name_field]} (~{highest['pop_sum_round']:,}); Lowest ADM2: {lowest[name_field]} (~{lowest['pop_sum_round']:,}).")
print("Wrote CSV + JSON and produced a choropleth and ranked bar chart.\n")


Inputs:
 - boundary: ZMB_adm2_gadm41_Copperbelt.shp | features: 12 | CRS: EPSG:4326
 - population raster: zmb_ppp_2020_constrained.tif | CRS: EPSG:4326 | nodata: -99999.0 | dtype: float32
 - all_touched: False

Results (using boundary as provided):
 - Highest population ADM2: Kitwe | pop_sum ≈ 769,410
 - Lowest population ADM2 : Mushindano | pop_sum ≈ 31,571
 - Sum over ADM2 polygons in file: ≈ 2,664,983

Saved:
 - C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\03_pop_by_adm2.csv
 - C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\03_pop_by_adm2.json

Saved map: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\03_pop_choropleth.png
Saved bar chart: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outp